In [2]:
import sys
import pandas as pd, numpy as np
from src.features import FEATURES
from src.metrics import report
val = pd.read_parquet("data/processed/features_val.parquet")
rows = val[(val.Open == 1) & (val.Sales > 0)] # real, open, known-sales days
train_df = rows[rows.Date <= "2015-06-18"]
valid_df = rows[(rows.Date >= "2015-06-19") & (rows.Date <= "2015-07-31")]
print(len(train_df), len(valid_df))

802942 41396


In [3]:
results = []
# Baseline 0: one global average
results.append(report("Global mean", valid_df.Sales,
np.full(len(valid_df), train_df.Sales.mean())))
# Baseline A: store x weekday average over the last 8 weeks before the cutoff
recent = train_df[train_df.Date > "2015-04-23"]
base = (recent.groupby(["Store", "DayOfWeek"])["Sales"].mean()
.rename("predA").reset_index())
v = valid_df.merge(base, on=["Store", "DayOfWeek"], how="left")
v["predA"] = v["predA"].fillna(train_df.Sales.mean())
results.append(report("Store-weekday avg (last 8 wks)", v.Sales, v.predA))
# Baseline B: same weekday last year (364 days ago), fallback to baseline A
v["predB"] = np.expm1(v["lag_364"]).fillna(v["predA"])
results.append(report("Same day last year", v.Sales, v.predB))
pd.DataFrame(results)

,model,RMSPE,MAPE_%,MAE
0,Global mean,0.5773,38.16,2242.2
1,Store-weekday avg (last 8 wks),0.2666,20.25,1284.3
2,Same day last year,0.1758,12.35,843.5
